# IEMOCAP zero-shot audio-LLM baseline (Kaggle)

**What this notebook produces:** one legitimate, reproducible **zero-shot** Speech Emotion
Recognition result on **IEMOCAP**, utterance-level, **audio + reference transcript**, with a
group-aligned audio-LLM.

| | |
|---|---|
| **Primary model** | `Qwen/Qwen2.5-Omni-3B` (zero-shot, no fine-tuning) — ingests the raw `.wav` **and** the transcript |
| **Fallback model** | `Qwen/Qwen2-Audio-7B-Instruct` (one-line switch) — also Qwen-family audio+text |
| **Secondary (optional)** | repo-native `Meta-Llama-3-8B-Instruct`, **text-only** (SpeechCueLLM prompt) |
| **Dataset / split** | IEMOCAP **Session 5 held-out test** (speaker-independent). Sessions 1–4 unused. **No cross-validation** — single fixed split, matching the FYP repo & its Final Report §4.1.1 |
| **Classes** | 6: `happy, sad, neutral, angry, excited, frustrated` — **happy and excited are NOT merged** |
| **N utterances** | **1622** (`xxx`/`sur`/`fea`/`oth`/`dis` dropped) |
| **Transcript** | IEMOCAP **ground-truth** reference transcription (not ASR) |
| **Metrics** | WA (accuracy), UA (unweighted recall), **macro-F1**, per-class P/R/F1, confusion matrix + the repo's own `report_score` verbatim |
| **Scoring code** | `match_text` / `optimize_output` / `report_score` copied **verbatim** from `src/LLM_code/main.py` |

**Leakage controls:** no VAD, no acoustic side-features, no gold labels and no future
utterances ever enter the prompt. Optional dialogue history (`USE_HISTORY=True`) is
**transcript-only, past+current turns only**, and contains no labels.

**What this is NOT:** it is **not** the FYP repo's published **72.0 % macro-F1** — that number is
the collaborator's **LoRA fine-tuned** Llama-3 + full private audio-feature pipeline
(private gender/VAD/eGeMaPS checkpoints, SharePoint-restricted). This notebook cannot
reproduce that without those checkpoints. Treat this as a **provisional zero-shot baseline
from a group-aligned model**, not a benchmark-final number.


## 0 · Kaggle setup (do this before running)

1. **Add data:** *Add Input* → your uploaded dataset (folder `kaggle_prep/kaggle_upload/`
   containing `test.json`, `test_tiny.json`, `test_with_history.json`, `manifest.json`,
   and `audio/` with 1622 wavs). ~240 MB.
2. **Accelerator:** *Settings → Accelerator →* **GPU T4 x2** (recommended). Single T4/P100 also works (auto-detected).
3. **Internet:** *Settings →* **Internet: On** (needed to download the model weights).
4. **Secrets (optional):** *Add-ons → Secrets →* add **`HF_TOKEN`** = a Hugging Face read token.
   Qwen models are **ungated**, so this is only needed for the optional Llama secondary run.
5. Run cells **top to bottom**. If a `transformers` version error appears right after the
   pip cell, use *Run → Restart & Run All* once (the pip cache makes the 2nd pass fast).
6. Nothing else to install or configure. The smoke cell (36 utts, ~1–3 min) runs first;
   glance at its raw generations, then the full 1622-utt cell runs (~1–2.5 h, resume-safe).


In [2]:
# ============================ CONFIG ============================
MODEL_FAMILY   = "qwen2.5-omni"     # "qwen2.5-omni" | "qwen2-audio" | "llama-text"
MODEL_ID       = None               # None -> family default (see below)

USE_HISTORY    = False              # True = transcript-only dialogue scaffold in the prompt
RUN_SMOKE_TEST = True               # run the 36-row tiny set first
RUN_FULL       = True               # then run the full 1622-row test set
FORCE_FULL     = False              # True = run full even if the smoke test looked broken

LIMIT          = None               # None = all 1622; or an int for a partial full-run
MAX_NEW_TOKENS = 12
FORCE_FP16     = None               # None = auto (fp16 on T4, else bf16); True/False to override
LOAD_IN_4BIT   = False              # only for llama-text / qwen2-audio on a single small GPU
SEED           = 11                 # matches repo SEED

OUTPUT_DIR     = "/kaggle/working/results"
# ==============================================================

FAMILY_DEFAULT_MODEL = {
    "qwen2.5-omni": "Qwen/Qwen2.5-Omni-3B",
    "qwen2-audio":  "Qwen/Qwen2-Audio-7B-Instruct",
    "llama-text":   "meta-llama/Meta-Llama-3-8B-Instruct",   # gated; see LLAMA_FALLBACK below
}
LLAMA_FALLBACK = "NousResearch/Meta-Llama-3-8B-Instruct"      # ungated mirror, identical weights
if MODEL_ID is None:
    MODEL_ID = FAMILY_DEFAULT_MODEL[MODEL_FAMILY]
print("MODEL_FAMILY :", MODEL_FAMILY)
print("MODEL_ID     :", MODEL_ID)
print("USE_HISTORY  :", USE_HISTORY)


MODEL_FAMILY : qwen2.5-omni
MODEL_ID     : Qwen/Qwen2.5-Omni-3B
USE_HISTORY  : False


In [3]:
import subprocess,sys

subprocess.run([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.52.3",
    "accelerate>=0.34",
    "qwen-omni-utils==0.0.4",
    "librosa==0.11.0",
    "soundfile==0.13.1",
    "av"
],check=True)

print("INSTALL DONE")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.6 MB/s eta 0:00:00
INSTALL DONE


In [4]:
import os, json, time, random, gc
import numpy as np
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import transformers
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA        :", torch.cuda.is_available(),
      "| GPUs:", torch.cuda.device_count(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"))
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("compute cap :", cap)
    _is_t4 = cap[0] == 7           # T4/V100 -> prefer fp16
else:
    _is_t4 = False
USE_FP16 = _is_t4 if FORCE_FP16 is None else FORCE_FP16
DTYPE = torch.float16 if USE_FP16 else torch.bfloat16
print("dtype       :", DTYPE)
os.makedirs(OUTPUT_DIR, exist_ok=True)


torch       : 2.10.0+cu128
transformers: 4.52.3
CUDA        : True | GPUs: 2 | Tesla T4
compute cap : (7, 5)
dtype       : torch.float16


In [5]:
# ---- locate the uploaded dataset ----
CANDIDATE_ROOTS = []
for base in ["/kaggle/input"]:
    if os.path.isdir(base):
        for d in sorted(os.listdir(base)):
            CANDIDATE_ROOTS.append(os.path.join(base, d))

def _find_input_dir():
    for root in CANDIDATE_ROOTS:
        for dirpath, _, files in os.walk(root):
            if "test.json" in files and os.path.isdir(os.path.join(dirpath, "audio")):
                return dirpath
    # allow test.json without audio/ only for llama-text
    for root in CANDIDATE_ROOTS:
        for dirpath, _, files in os.walk(root):
            if "test.json" in files:
                return dirpath
    raise FileNotFoundError(
        "Could not find test.json (+ audio/) under /kaggle/input. "
        "Add your uploaded dataset via 'Add Input'. Seen: " + str(CANDIDATE_ROOTS))

INPUT_DIR = _find_input_dir()
AUDIO_DIR = os.path.join(INPUT_DIR, "audio")
print("INPUT_DIR:", INPUT_DIR)
print("has audio/:", os.path.isdir(AUDIO_DIR),
      "| n wav:", len(os.listdir(AUDIO_DIR)) if os.path.isdir(AUDIO_DIR) else 0)
with open(os.path.join(INPUT_DIR, "manifest.json"), encoding="utf-8") as f:
    MANIFEST = json.load(f)
print("\n--- dataset manifest (provenance) ---")
print("repo_commit        :", MANIFEST["repo_commit"])
print("split_protocol     :", MANIFEST["split_protocol"])
print("taxonomy           :", MANIFEST["taxonomy"], "| merged happy+excited:", MANIFEST["happy_excited_merged"])
print("eval rows          :", MANIFEST["counts"]["session5_6class_eval_rows"])
print("per class          :", MANIFEST["counts"]["session5_6class_per_class"])


INPUT_DIR: /kaggle/input/datasets/pranavjaiganesh/dataset1/kaggle_upload
has audio/: True | n wav: 1622

--- dataset manifest (provenance) ---
repo_commit        : dd91645989d1aff6faa259da4a3eb874fe6a19a9
split_protocol     : speaker-independent; Sessions 1-4 = train, Session 5 = test (repo: df['split']=np.where(df['session'].between(1,4),'train','test'))
taxonomy           : 6-class: happy, sad, neutral, angry, excited, frustrated | merged happy+excited: False
eval rows          : 1622
per class          : {'happy': 143, 'sad': 245, 'neutral': 384, 'angry': 170, 'excited': 299, 'frustrated': 381}


In [6]:
# ---- HF auth (optional; Qwen is ungated) ----
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen):", e)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as e:
        print("hf login skipped:", e)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF_TOKEN loaded from Kaggle Secrets.


In [7]:
# ================= iemocap_eval_lib (scoring + prompt) =================
# The functions between "VERBATIM" markers are copied unchanged from
# src/LLM_code/main.py so scoring is byte-identical to the FYP repo.
_LIB = r"""
import json
import numpy as np
from sklearn import metrics
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# ===== VERBATIM from src/LLM_code/main.py =====
def get_labels_attr(dataset):
    label_list_set = {'iemocap':['happy','sad','neutral','angry','excited','frustrated'],
        'msp':["angry","frustrated","disgust","annoyed","sad","depressed","disappointed","fear","happy","surprise","excited","contempt","amused","concerned","confused","neutral"]}
    label_str_set = {'iemocap':"'happy', 'sad', 'neutral', 'angry', 'excited', 'frustrated'",
        'msp':"'angry', 'frustrated', 'disgust', 'annoyed', 'sad', 'depressed', 'disappointed', 'fear', 'happy', 'surprise', 'excited', 'contempt', 'amused', 'concerned', 'confused', 'neutral'"}
    labels = label_list_set[dataset]
    if 'unknown' not in labels: labels.append('unknown')
    emotional_label_dict = {t:n for n,t in enumerate(labels)}
    return emotional_label_dict, label_str_set[dataset]

def report_score(dataset, golds, preds, mode='test'):
    if dataset == 'iemocap':
        target_names = ['hap','sad','neu','ang','exc','fru','unknown']; digits = 7
    else:
        target_names = ["angry","frustrated","disgust","annoyed","sad","depressed","disappointed","fear","happy","surprise","excited","contempt","amused","concerned","confused","neutral","unknown"]; digits = 17
    res = {}
    res['Acc_SA'] = accuracy_score(golds, preds)
    res['F1_SA'] = f1_score(golds, preds, average='weighted')
    res['mode'] = mode
    for k,v in res.items():
        if isinstance(v, float): res[k] = round(v*100, 3)
    res_matrix = metrics.classification_report(golds, preds, labels=list(range(len(target_names))), target_names=target_names, digits=digits, zero_division=0)
    return res, res_matrix

def match_text(text, word_set_):
    if text is None: return []
    len_text = len(text); s_idx = 0; match_res = []
    while s_idx < len_text:
        cache = []; span_length = 1
        while span_length < 12 and s_idx + span_length <= len_text:
            span = text[s_idx: s_idx + span_length]
            if span in word_set_: cache.append(span)
            span_length += 1
        if len(cache) > 0:
            match_res.append(cache[-1]); s_idx += len(cache[-1])
        else: s_idx += 1
    return match_res

def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = dp[i-1][j-1] if s1[i-1]==s2[j-1] else min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1]) + 1
    return dp[m][n]

def optimize_output(output, label_set):
    min_distance = float('inf'); optimized_output = None
    for label in label_set:
        d = edit_distance(output, label)
        if d < min_distance: min_distance = d; optimized_output = label
    return optimized_output
# ===== END VERBATIM =====

IEMOCAP_LABELS = ['happy','sad','neutral','angry','excited','frustrated']
IEMOCAP_LABEL_SET_STR = 'happy, sad, neutral, angry, excited, frustrated'

def map_answer_to_id(answer, emotional_label_dict):
    valid = [k for k in emotional_label_dict.keys() if k != 'unknown']
    unknown_id = emotional_label_dict.get('unknown', len(emotional_label_dict)-1)
    m = match_text(answer, valid)
    if m: return emotional_label_dict[m[0]], False
    opt = optimize_output(answer, valid)
    return emotional_label_dict.get(opt, unknown_id), True

def build_prompt(utterance, history_context=None):
    if history_context:
        return ("Now you are expert of sentiment and emotional analysis.\n"
                "The following conversation noted between '### ###' involves several speakers. "
                "The last utterances are the dialogue context for the target. ### "
                f"{history_context}"
                " ###\n"
                f'Target transcript: \"{utterance}\"\n'
                "You are also given the target utterance audio. "
                f"Please select the emotional label of the target from <{IEMOCAP_LABEL_SET_STR}> "
                "based on both the transcript and the audio. Respond with just one label:")
    return ("Now you are expert of sentiment and emotional analysis.\n"
            "You are given one spoken utterance: its audio and its transcript.\n"
            f'Transcript: \"{utterance}\"\n'
            f"Please select the emotional label of the utterance from <{IEMOCAP_LABEL_SET_STR}> "
            "based on both the transcript and the audio. Respond with just one label:")

def build_prompt_repo_iemocap(utterance, history_context=None):
    # VERBATIM template from src/LLM_code/main.py DynamicPromptCollator (iemocap branch,
    # L593-605), with description_str='' (acoustic-feature categories require the private
    # gender/VAD/eGeMaPS checkpoints and are omitted). Used only for MODEL_FAMILY='llama-text'
    # so that path is a faithful repo-native text-only zero-shot reproduction.
    convo_history = history_context if history_context else "No context available."
    description_str = ""
    return (
        "Now you are expert of sentiment and emotional analysis.\n"
        "The following conversation noted between '### ###' involves several speakers. "
        "The last three utterances are followed by its speech features. ### "
        f"{convo_history}"
        " ###\n"
        "Target speech characteristics:\n"
        f"{description_str}\n"
        f'Transcript: \"{utterance}\"\n'
        f"Please select the emotional label of the transcript from <{IEMOCAP_LABEL_SET_STR}> "
        "based on both the context and audio features. Respond with just one label:"
    )

def score_predictions(records, raw_answers, dataset='iemocap'):
    eld, _ = get_labels_attr(dataset)
    golds, preds, confuse, per_row = [], [], [], []
    inv = {v:k for k,v in eld.items()}
    for i, ans in enumerate(raw_answers):
        gw = records[i]['output']
        g = eld.get(gw, eld.get('unknown', len(eld)-1))
        p, isc = map_answer_to_id(ans, eld)
        golds.append(g); preds.append(p)
        if isc: confuse.append(i)
        per_row.append({"id":records[i]["id"], "gold":gw, "raw_generation":ans,
                        "pred":inv[p], "edit_distance_fallback":isc})
    repo_res, repo_matrix = report_score(dataset, golds, preds)
    real = list(range(len(IEMOCAP_LABELS)))
    g = np.array(golds); pr = np.array(preds)
    acc = accuracy_score(g, pr)
    macro_f1 = f1_score(g, pr, labels=real, average='macro', zero_division=0)
    weighted_f1 = f1_score(g, pr, labels=real, average='weighted', zero_division=0)
    ua = metrics.recall_score(g, pr, labels=real, average='macro', zero_division=0)
    p_c, r_c, f_c, s_c = metrics.precision_recall_fscore_support(g, pr, labels=real, zero_division=0)
    per_class = {IEMOCAP_LABELS[k]:{"precision":round(float(p_c[k])*100,3),
        "recall":round(float(r_c[k])*100,3),"f1":round(float(f_c[k])*100,3),
        "support":int(s_c[k])} for k in real}
    cm = confusion_matrix(g, pr, labels=real).tolist()
    rep = classification_report(g, pr, labels=real, target_names=IEMOCAP_LABELS, digits=4, zero_division=0)
    return {"n_samples":len(records), "n_edit_distance_fallback":len(confuse),
            "accuracy_WA":round(float(acc)*100,3), "UA_unweighted_recall":round(float(ua)*100,3),
            "macro_f1":round(float(macro_f1)*100,3), "weighted_f1":round(float(weighted_f1)*100,3),
            "per_class":per_class,
            "confusion_matrix":{"labels":IEMOCAP_LABELS,"rows_gold_cols_pred":cm},
            "sklearn_classification_report_6class":rep,
            "repo_report_score":repo_res,
            "repo_classification_report_7class":repo_matrix,
            "label_id_map":eld, "predictions":per_row}
"""
with open("/kaggle/working/iemocap_eval_lib.py", "w", encoding="utf-8") as f:
    f.write(_LIB)
import importlib, sys
sys.path.insert(0, "/kaggle/working")
import iemocap_eval_lib as EVAL
importlib.reload(EVAL)
print("eval lib ready. labels:", EVAL.IEMOCAP_LABELS)
print("\nExample prompt:\n", EVAL.build_prompt('I can not believe you did that.'))


eval lib ready. labels: ['happy', 'sad', 'neutral', 'angry', 'excited', 'frustrated']

Example prompt:
 Now you are expert of sentiment and emotional analysis.
You are given one spoken utterance: its audio and its transcript.
Transcript: "I can not believe you did that."
Please select the emotional label of the utterance from <happy, sad, neutral, angry, excited, frustrated> based on both the transcript and the audio. Respond with just one label:


In [8]:
# ---- records ----
def load_records(name):
    with open(os.path.join(INPUT_DIR, name), encoding="utf-8") as f:
        return json.load(f)

TINY = load_records("test_tiny_with_history.json" if USE_HISTORY else "test_tiny.json")
FULL = load_records("test_with_history.json" if USE_HISTORY else "test.json")
if LIMIT:
    FULL = FULL[:LIMIT]
print(f"tiny: {len(TINY)} rows | full: {len(FULL)} rows | USE_HISTORY={USE_HISTORY}")
print("tiny gold dist:", {w: sum(1 for r in TINY if r['output']==w) for w in EVAL.IEMOCAP_LABELS})


tiny: 36 rows | full: 1622 rows | USE_HISTORY=False
tiny gold dist: {'happy': 6, 'sad': 6, 'neutral': 6, 'angry': 6, 'excited': 6, 'frustrated': 6}


In [9]:
# ================= model loading =================
_MODEL = {}   # holds handles

def load_model():
    fam = MODEL_FAMILY
    if fam == "qwen2.5-omni":
        import transformers as _tf
        from transformers import Qwen2_5OmniProcessor
        from qwen_omni_utils import process_mm_info
        OmniCls = getattr(_tf, "Qwen2_5OmniForConditionalGeneration",
                          getattr(_tf, "Qwen2_5OmniModel", None))
        if OmniCls is None:
            raise ImportError(
                "This transformers build has no Qwen2.5-Omni class. "
                "Set MODEL_FAMILY='qwen2-audio' and re-run.")
        proc = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
        try:            # skip loading the speech 'talker' entirely (~2 GB) - text only
            model = OmniCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE,
                                            device_map="auto", enable_audio_output=False)
        except TypeError:
            model = OmniCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
        for _m in ("disable_talker",):
            if hasattr(model, _m):
                try: getattr(model, _m)(); print(_m, "done")
                except Exception as e: print(_m, "skipped:", e)
        model.eval()
        _MODEL.update(model=model, proc=proc, process_mm_info=process_mm_info)

    elif fam == "qwen2-audio":
        from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor
        import librosa
        proc = AutoProcessor.from_pretrained(MODEL_ID)
        kw = dict(torch_dtype=DTYPE, device_map="auto")
        if LOAD_IN_4BIT:
            from transformers import BitsAndBytesConfig
            kw = dict(device_map="auto", quantization_config=BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True))
        model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, **kw).eval()
        _MODEL.update(model=model, proc=proc, librosa=librosa,
                      sr=proc.feature_extractor.sampling_rate)

    elif fam == "llama-text":
        from transformers import AutoModelForCausalLM, AutoTokenizer
        mid = MODEL_ID
        try:
            tok = AutoTokenizer.from_pretrained(mid, token=HF_TOKEN)
        except Exception as e:
            print(f"{mid} not accessible ({e}); falling back to {LLAMA_FALLBACK}")
            mid = LLAMA_FALLBACK
            tok = AutoTokenizer.from_pretrained(mid)
        kw = dict(torch_dtype=DTYPE, device_map="auto", token=HF_TOKEN)
        if LOAD_IN_4BIT:
            from transformers import BitsAndBytesConfig
            kw = dict(device_map="auto", token=HF_TOKEN,
                      quantization_config=BitsAndBytesConfig(
                        load_in_4bit=True, bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True))
        model = AutoModelForCausalLM.from_pretrained(mid, **kw).eval()
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        tok.padding_side = "left"
        _MODEL.update(model=model, tok=tok, resolved_id=mid)
    else:
        raise ValueError(MODEL_FAMILY)
    print("model loaded:", MODEL_ID)

load_model()


preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

spk_dict.pt:   0%|          | 0.00/260k [00:00<?, ?B/s]

disable_talker done
model loaded: Qwen/Qwen2.5-Omni-3B


In [10]:
# ================= per-utterance inference =================
QWEN_SYS = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
            "capable of perceiving auditory and visual inputs, as well as generating text and speech.")

@torch.no_grad()
def infer_one(rec):
    prompt = EVAL.build_prompt(rec["utterance"], rec.get("history_context") if USE_HISTORY else None)
    wav_path = os.path.join(INPUT_DIR, rec["path"]) if "path" in rec else os.path.join(AUDIO_DIR, rec["id"] + ".wav")
    fam = MODEL_FAMILY

    if fam == "qwen2.5-omni":
        model, proc, process_mm_info = _MODEL["model"], _MODEL["proc"], _MODEL["process_mm_info"]
        conv = [
            {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
            {"role": "user", "content": [
                {"type": "audio", "audio": wav_path},
                {"type": "text", "text": prompt}]},
        ]
        text = proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
        try:
            audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
        except TypeError:
            audios, images, videos = process_mm_info(conv)
        try:
            inputs = proc(text=text, audio=audios, images=images, videos=videos,
                          return_tensors="pt", padding=True)
        except TypeError:
            inputs = proc(text=text, audios=audios, images=images, videos=videos,
                          return_tensors="pt", padding=True)
        inputs = inputs.to(model.device)
        try:
            inputs = inputs.to(model.dtype)          # casts float feats only; ids stay long
        except Exception:
            pass
        gkw = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        try:
            out = model.generate(**inputs, return_audio=False, **gkw)
        except TypeError:
            out = model.generate(**inputs, **gkw)
        if isinstance(out, (tuple, list)):
            out = out[0]
        gen = out[:, inputs["input_ids"].shape[1]:]
        return proc.batch_decode(gen, skip_special_tokens=True,
                                 clean_up_tokenization_spaces=False)[0].strip()

    if fam == "qwen2-audio":
        model, proc, librosa, sr = (_MODEL[k] for k in ("model", "proc", "librosa", "sr"))
        conv = [{"role": "user", "content": [
                    {"type": "audio", "audio_url": wav_path},
                    {"type": "text", "text": prompt}]}]
        text = proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
        wav, _ = librosa.load(wav_path, sr=sr)
        inputs = proc(text=text, audios=[wav], sampling_rate=sr, return_tensors="pt", padding=True)
        inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        gen = out[:, inputs["input_ids"].shape[1]:]
        return proc.batch_decode(gen, skip_special_tokens=True)[0].strip()

    if fam == "llama-text":
        model, tok = _MODEL["model"], _MODEL["tok"]
        # repo-native (text-only) => use the repo's exact prompt template
        prompt = EVAL.build_prompt_repo_iemocap(
            rec["utterance"], rec.get("history_context") if USE_HISTORY else None)
        msgs = [{"role": "user", "content": prompt}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
        out = model.generate(ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

# quick single-shot check
t0 = time.time()
_demo = infer_one(TINY[0])
print(f"1st inference OK in {time.time()-t0:.1f}s  ->  id={TINY[0]['id']}  gold={TINY[0]['output']}  raw={_demo!r}")


1st inference OK in 14.0s  ->  id=Ses05F_impro01_F000  gold=neutral  raw='neutral'


In [11]:
# ================= SMOKE TEST (36 rows) =================
from tqdm.auto import tqdm
def run_set(records, tag):
    raws, t0 = [], time.time()
    for r in tqdm(records, desc=tag):
        try:
            raws.append(infer_one(r))
        except Exception as e:
            print("infer error on", r["id"], "->", repr(e)); raws.append("")
    dt = time.time() - t0
    print(f"{tag}: {len(records)} rows in {dt:.1f}s ({dt/max(len(records),1):.2f}s/row)")
    return raws

SMOKE_OK = None
if RUN_SMOKE_TEST:
    tiny_raw = run_set(TINY, "smoke")
    tiny_metrics = EVAL.score_predictions(TINY, tiny_raw)
    for pr in tiny_metrics["predictions"]:
        print(f'  {pr["id"]:<24} gold={pr["gold"]:<10} pred={pr["pred"]:<10} '
              f'{"(editdist)" if pr["edit_distance_fallback"] else ""}  raw={pr["raw_generation"]!r}')
    n_empty = sum(1 for x in tiny_raw if not x.strip())
    n_fb = tiny_metrics["n_edit_distance_fallback"]
    SMOKE_OK = (n_empty <= 3) and (n_fb <= len(TINY) // 2)
    print("\nsmoke metrics: WA=%.2f  macro-F1=%.2f  UA=%.2f  | empty=%d/%d  editdist-fallback=%d/%d" % (
        tiny_metrics["accuracy_WA"], tiny_metrics["macro_f1"],
        tiny_metrics["UA_unweighted_recall"],
        n_empty, len(TINY), n_fb, len(TINY)))
    print(f"\nSMOKE_OK = {SMOKE_OK}  "
          + ("-> proceed to the full run." if SMOKE_OK else
             "-> generations look broken (empty/garbled). Fix the config "
             "(try MODEL_FAMILY='qwen2-audio') before the full run, or set FORCE_FULL=True to override."))
else:
    print("smoke test skipped")


smoke:   0%|          | 0/36 [00:00<?, ?it/s]

smoke: 36 rows in 17.8s (0.49s/row)
  Ses05F_impro01_F000      gold=neutral    pred=neutral      raw='neutral'
  Ses05F_impro01_M000      gold=neutral    pred=neutral      raw='neutral'
  Ses05F_impro01_F001      gold=frustrated pred=frustrated   raw='frustrated'
  Ses05F_impro01_F002      gold=frustrated pred=neutral      raw='neutral'
  Ses05F_impro01_F003      gold=frustrated pred=neutral      raw='neutral'
  Ses05F_impro01_M003      gold=neutral    pred=frustrated   raw='frustrated'
  Ses05F_impro01_F004      gold=frustrated pred=frustrated (editdist)  raw='Frustrated.'
  Ses05F_impro01_M004      gold=frustrated pred=neutral    (editdist)  raw='Neutral. What do you think about this?'
  Ses05F_impro01_F005      gold=frustrated pred=neutral      raw='neutral'
  Ses05F_impro01_F008      gold=neutral    pred=neutral    (editdist)  raw='Neutral'
  Ses05F_impro01_F017      gold=angry      pred=frustrated   raw='frustrated'
  Ses05F_impro01_F018      gold=angry      pred=neutral      raw=

In [12]:
# ================= FULL RUN (1622 rows) — incremental + resume-safe =================
PRED_JSONL = os.path.join(OUTPUT_DIR, f"pred_{MODEL_FAMILY}_{'hist' if USE_HISTORY else 'nohist'}.jsonl")

def load_done():
    done = {}
    if os.path.exists(PRED_JSONL):
        with open(PRED_JSONL, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line: continue
                o = json.loads(line); done[o["id"]] = o["raw"]
    return done

if RUN_FULL and RUN_SMOKE_TEST and (SMOKE_OK is False) and not FORCE_FULL:
    raise RuntimeError(
        "Smoke test looked broken (see cell above). Not starting the 1622-row run. "
        "Fix the config or set FORCE_FULL=True.")

if RUN_FULL:
    done = load_done()
    print(f"resuming: {len(done)} / {len(FULL)} already done")
    t0 = time.time()
    with open(PRED_JSONL, "a", encoding="utf-8") as f:
        for i, r in enumerate(tqdm(FULL, desc="full")):
            if r["id"] in done: continue
            try:
                raw = infer_one(r)
            except Exception as e:
                print("infer error", r["id"], repr(e)); raw = ""
            done[r["id"]] = raw
            f.write(json.dumps({"id": r["id"], "raw": raw}, ensure_ascii=False) + "\n")
            f.flush()
            if (i+1) % 100 == 0:
                el = time.time() - t0
                print(f"  {i+1}/{len(FULL)}  elapsed {el/60:.1f}m  eta {el/(i+1)*(len(FULL)-i-1)/60:.1f}m")
    print(f"full run done in {(time.time()-t0)/60:.1f} min -> {PRED_JSONL}")
else:
    print("full run skipped")


resuming: 0 / 1622 already done


full:   0%|          | 0/1622 [00:00<?, ?it/s]

  100/1622  elapsed 0.8m  eta 12.0m
  200/1622  elapsed 1.6m  eta 11.3m
  300/1622  elapsed 2.4m  eta 10.5m
  400/1622  elapsed 3.2m  eta 9.7m
  500/1622  elapsed 4.0m  eta 9.0m
  600/1622  elapsed 4.8m  eta 8.1m
  700/1622  elapsed 5.6m  eta 7.3m
  800/1622  elapsed 6.4m  eta 6.5m
  900/1622  elapsed 7.2m  eta 5.8m
  1000/1622  elapsed 8.0m  eta 5.0m
  1100/1622  elapsed 8.8m  eta 4.2m
  1200/1622  elapsed 9.5m  eta 3.3m
  1300/1622  elapsed 10.3m  eta 2.6m
  1400/1622  elapsed 11.1m  eta 1.8m
  1500/1622  elapsed 11.9m  eta 1.0m
  1600/1622  elapsed 12.7m  eta 0.2m
full run done in 12.9 min -> /kaggle/working/results/pred_qwen2.5-omni_nohist.jsonl


In [13]:
# ================= SCORE + SAVE =================
import pandas as pd
done = {}
with open(PRED_JSONL, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            o = json.loads(line); done[o["id"]] = o["raw"]

scored = [r for r in FULL if r["id"] in done]
missing = [r["id"] for r in FULL if r["id"] not in done]
if missing:
    print(f"WARNING: {len(missing)} rows have no prediction (timeout?). Scoring the {len(scored)} available. "
          f"Re-run the full-run cell to finish, then re-run this cell.")
raws = [done[r["id"]] for r in scored]

M = EVAL.score_predictions(scored, raws)

MODE = {"qwen2.5-omni":"BASE-MODEL ZERO-SHOT (audio + transcript)",
        "qwen2-audio":"BASE-MODEL ZERO-SHOT (audio + transcript)",
        "llama-text":"BASE-MODEL ZERO-SHOT (transcript only; no audio)"}[MODEL_FAMILY]
resolved_id = _MODEL.get("resolved_id", MODEL_ID)

banner = []
banner.append("="*78)
banner.append("IEMOCAP zero-shot baseline — RESULT")
banner.append("="*78)
banner.append(f"model                : {resolved_id}")
banner.append(f"mode                 : {MODE}   [NOT fine-tuned; no private checkpoint]")
banner.append(f"dataset / split      : IEMOCAP Session 5 held-out test (speaker-independent)")
banner.append(f"cross-validation     : none (single fixed split, matches FYP repo)")
banner.append(f"emotion classes (6)  : {', '.join(EVAL.IEMOCAP_LABELS)}")
banner.append(f"happy + excited merged: NO")
banner.append(f"transcript source    : IEMOCAP ground-truth reference (not ASR)")
banner.append(f"dialogue history     : {'transcript-only, window=8, no labels' if USE_HISTORY else 'none (per-utterance)'}")
banner.append(f"utterances evaluated : {M['n_samples']}" + (f"  (+{len(missing)} missing)" if missing else "  (full 1622)" if M['n_samples']==1622 else ""))
banner.append(f"edit-distance fallbacks: {M['n_edit_distance_fallback']} / {M['n_samples']}  (unparseable generations)")
banner.append("-"*78)
banner.append(f"Accuracy / WA        : {M['accuracy_WA']:.2f} %")
banner.append(f"UA (unweighted recall): {M['UA_unweighted_recall']:.2f} %")
banner.append(f"macro-F1  (6 classes): {M['macro_f1']:.2f} %   <-- headline metric")
banner.append(f"weighted-F1 (6 classes): {M['weighted_f1']:.2f} %")
banner.append("-"*78)
banner.append("per-class (precision / recall / F1 / support):")
for k, v in M["per_class"].items():
    banner.append(f"  {k:<11} {v['precision']:6.2f} / {v['recall']:6.2f} / {v['f1']:6.2f}   (n={v['support']})")
banner.append("-"*78)
banner.append("confusion matrix (rows=gold, cols=pred; order = " + ", ".join(EVAL.IEMOCAP_LABELS) + "):")
for lbl, row in zip(EVAL.IEMOCAP_LABELS, M["confusion_matrix"]["rows_gold_cols_pred"]):
    banner.append(f"  {lbl:<11} " + " ".join(f"{x:4d}" for x in row))
banner.append("-"*78)
banner.append("repo report_score (verbatim src/LLM_code/main.py; F1_SA is weighted, 7-class incl 'unknown'):")
banner.append(f"  {M['repo_report_score']}")
banner.append("")
banner.append(M["repo_classification_report_7class"])
banner.append("="*78)
banner.append("NOTE: this is a provisional zero-shot baseline. It is NOT the FYP repo's published")
banner.append("72.0% macro-F1 (that requires the collaborator's LoRA fine-tuned checkpoint +")
banner.append("private audio-feature pipeline).")
report_txt = "\n".join(banner)
print(report_txt)

# ---- save artifacts ----
tag = f"{MODEL_FAMILY}_{'hist' if USE_HISTORY else 'nohist'}"
with open(os.path.join(OUTPUT_DIR, f"metrics_{tag}.json"), "w", encoding="utf-8") as f:
    json.dump({k: v for k, v in M.items() if k != "predictions"}, f, ensure_ascii=False, indent=2)
with open(os.path.join(OUTPUT_DIR, f"predictions_{tag}.json"), "w", encoding="utf-8") as f:
    json.dump(M["predictions"], f, ensure_ascii=False, indent=2)
pd.DataFrame(M["confusion_matrix"]["rows_gold_cols_pred"],
             index=EVAL.IEMOCAP_LABELS, columns=EVAL.IEMOCAP_LABELS
             ).to_csv(os.path.join(OUTPUT_DIR, f"confusion_{tag}.csv"))
with open(os.path.join(OUTPUT_DIR, f"report_{tag}.txt"), "w", encoding="utf-8") as f:
    f.write(report_txt + "\n")
run_meta = {"model_id": resolved_id, "model_family": MODEL_FAMILY, "mode": MODE,
            "use_history": USE_HISTORY, "limit": LIMIT, "n_scored": M["n_samples"],
            "n_missing": len(missing), "dtype": str(DTYPE),
            "transformers": transformers.__version__, "torch": torch.__version__,
            "dataset_manifest": MANIFEST}
with open(os.path.join(OUTPUT_DIR, f"run_meta_{tag}.json"), "w", encoding="utf-8") as f:
    json.dump(run_meta, f, ensure_ascii=False, indent=2)
print("\nsaved to", OUTPUT_DIR, "->", sorted(os.listdir(OUTPUT_DIR)))


IEMOCAP zero-shot baseline — RESULT
model                : Qwen/Qwen2.5-Omni-3B
mode                 : BASE-MODEL ZERO-SHOT (audio + transcript)   [NOT fine-tuned; no private checkpoint]
dataset / split      : IEMOCAP Session 5 held-out test (speaker-independent)
cross-validation     : none (single fixed split, matches FYP repo)
emotion classes (6)  : happy, sad, neutral, angry, excited, frustrated
happy + excited merged: NO
transcript source    : IEMOCAP ground-truth reference (not ASR)
dialogue history     : none (per-utterance)
utterances evaluated : 1622  (full 1622)
edit-distance fallbacks: 445 / 1622  (unparseable generations)
------------------------------------------------------------------------------
Accuracy / WA        : 38.78 %
UA (unweighted recall): 38.10 %
macro-F1  (6 classes): 34.21 %   <-- headline metric
weighted-F1 (6 classes): 33.09 %
------------------------------------------------------------------------------
per-class (precision / recall / F1 / support):
  hap

## What this run legitimately produces

A **zero-shot** IEMOCAP Session-5 (speaker-independent, 6-class, 1622-utterance)
emotion-recognition score for a **group-aligned audio-LLM** (`Qwen2.5-Omni-3B` by
default), consuming **raw audio + ground-truth transcript**, scored with the FYP
repo's own `match_text` / `report_score` code and standard macro-F1 / WA / per-class /
confusion-matrix metrics. No training, no leakage.

## What remains impossible without the collaborator's ("Hoang Anh") private checkpoint

* The repo's **72.0 % macro-F1** headline number — that is the **LoRA fine-tuned
  Llama-3-8B** plus the private **gender classifier + VAD regressor + eGeMaPS**
  feature pipeline (`checkpoints/iemocap_checkpoints/{LLM,VAD_Regressor,Gender_Classifier}_checkpoint`),
  SharePoint-restricted.
* The repo's `main_llm.sh iemocap inference` and `run_inference.sh` paths — both hard-require
  `LLM_checkpoint` (a DeepSpeed ZeRO checkpoint) and error out without it.
* The repo's intended prompt with **text-encoded acoustic features** (pitch/loudness/VAD
  bins) — those categories come from the private audio models, so this baseline omits
  them and gives the audio to the model as a waveform instead.
* Any claim of parity with the NTU group benchmark leaderboard — this is one
  reproducible baseline data point, not a benchmark-final submission.
